# 🎵 AI Music Generation Studio — Google Colab

**Meta's MusicGen** running on a free T4 GPU.

### Steps
1. Runtime → Change runtime type → **T4 GPU** (free)
2. Run all cells in order
3. Use the Gradio link that appears at the end

---

## Step 1 — Check GPU

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## Step 2 — Install Dependencies

In [ ]:
%%capture
!pip install audiocraft gradio librosa soundfile matplotlib
print('✅ All dependencies installed')

## Step 3 — Clone the Project

In [ ]:
# Replace with your GitHub repo URL after uploading
REPO_URL = 'https://github.com/YOUR_USERNAME/CodeAlpha_MusicGenerationAI'

!git clone {REPO_URL} music-gen
%cd music-gen
print('✅ Repository ready')

## Step 4 — Quick Generate (no UI)

In [ ]:
import torch
from audiocraft.models import MusicGen
from audiocraft.data.audio import audio_write
from IPython.display import Audio, display

# ── Load model ────────────────────────────────────────────────────────────
model = MusicGen.get_pretrained('facebook/musicgen-small')
model.set_generation_params(duration=10, temperature=1.0, top_k=250, cfg_coef=3.0)

# ── Generate ──────────────────────────────────────────────────────────────
PROMPT = 'Upbeat jazz piano trio with walking bass and brushed snare'
print(f'Generating: {PROMPT}')

wav = model.generate([PROMPT], progress=True)

# ── Save & play ───────────────────────────────────────────────────────────
audio_write('output', wav[0].cpu(), model.sample_rate,
            strategy='loudness', loudness_compressor=True)

display(Audio('output.wav'))
print('✅ Saved to output.wav')

## Step 5 — Launch Full Studio UI

In [ ]:
import sys
sys.path.insert(0, '.')

from app import build_app

app = build_app()
app.launch(share=True)   # share=True gives a public Gradio URL

## Bonus — Melody-Conditioned Generation
Upload any audio file and use it as a melodic reference.

In [ ]:
import torchaudio
from google.colab import files
from IPython.display import Audio, display

# Upload your melody file
uploaded = files.upload()
melody_file = list(uploaded.keys())[0]

# Load melody
melody, sr = torchaudio.load(melody_file)
print(f'Melody: {melody_file} | SR: {sr} | Shape: {melody.shape}')

# Load melody model
model_melody = MusicGen.get_pretrained('facebook/musicgen-melody')
model_melody.set_generation_params(duration=10)

PROMPT = 'Jazz piano arrangement of this melody, with upright bass and light percussion'

wav = model_melody.generate_with_chroma(
    descriptions=[PROMPT],
    melody_wavs=melody.unsqueeze(0),
    melody_sample_rate=sr,
    progress=True,
)

audio_write('melody_output', wav[0].cpu(), model_melody.sample_rate,
            strategy='loudness', loudness_compressor=True)

display(Audio('melody_output.wav'))
print('✅ Melody-conditioned generation complete!')

## Tips

| Goal | Setting |
|---|---|
| More creative output | Temperature 1.2–1.5 |
| Closer to prompt | CFG coef 5–7 |
| Faster generation | Use `musicgen-small` |
| Best quality | Use `musicgen-large` (needs >8 GB VRAM) |
| Melody reference | Use `musicgen-melody` + upload audio |